# Dynamic Iceberg Tables Lab Guide

## What We Built So Far

In the **CLD Lab Guide**, you connected Snowflake to AWS Glue via a Catalog-Linked Database using **vended credentials**. Bronze balloon game events are now queryable in Snowflake — no data copy, no ETL.

## What We'll Build Next

This notebook creates **five Dynamic Iceberg Tables** that build a **silver-layer aggregation pipeline** over the bronze data. Each DIT:

- Reads raw JSON events from the bronze Iceberg table via the CLD
- Parses and aggregates them into a specific view (leaderboards, color stats, real-time scores, trends)
- Auto-refreshes on a declared **TARGET_LAG** schedule
- Writes Iceberg-format data to Snowflake Managed Storage — readable by any Iceberg-compatible engine (Spark, DuckDB, etc.)

**Prerequisites:**
- A Catalog-Linked Database (CLD) reading the bronze Iceberg table (from the CLD Lab Guide)
- USAGE privilege on the warehouse and external volume
- Role permissions to create Dynamic Tables and read the bronze source

---

## Step 1: Configure Your Environment

Set the variables below to match your environment. All subsequent SQL cells reference these variables via Jinja templating — change them once and everything updates.

> ### STOP — Update the variables below before proceeding!
>
> Make sure you have set **all five variables** to match your environment, then **run the cell below** before continuing.

In [ ]:
WAREHOUSE = 'DEFAULT_WH'
EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
TARGET_LAG = '5 minutes'
CLD_DATABASE = 'balloon_game_events'
GLUE_DB = 'summit26_ar103_balloon_pops'
GLUE_NAMESPACE = f"{GLUE_DB}"
BRONZE_ICEBERG_TABLE = f'{CLD_DATABASE}."{GLUE_NAMESPACE}"."balloon_game_events"'
DB_NAME = 'summit26_ar103_balloon_silver'

---

## Step 2: Create the Silver Database and Schema

Create a Snowflake-managed database and schema to hold the silver-layer Dynamic Iceberg Tables.

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;

In [ ]:
%%sql -r create_db_result
CREATE DATABASE IF NOT EXISTS {{DB_NAME}}
  COMMENT = 'Snowflake-managed silver (Dynamic Iceberg Tables over CLD bronze)';

In [ ]:
%%sql -r create_schema_result
CREATE SCHEMA IF NOT EXISTS {{DB_NAME}}.silver
  COMMENT = 'Aggregates from bronze balloon_game_events (JSON column event)';

---

## Step 2b: Set Notebook Context

Point the notebook session at the database and schema we just created so subsequent DDL runs in the right place.

In [ ]:
%%sql -r set_context_result
USE ROLE ACCOUNTADMIN;
USE DATABASE {{DB_NAME}};
USE SCHEMA silver;

---

## Step 3: Player Leaderboard (`dt_player_leaderboard`)

> **Why Dynamic Iceberg Tables?** ([docs](https://docs.snowflake.com/en/user-guide/dynamic-tables-create-iceberg))
>
> Dynamic Iceberg Tables combine Snowflake's declarative pipeline engine with the open Iceberg format:
>
> - **Declarative SQL** — define *what* to compute, not *when* or *how* to refresh
> - **TARGET_LAG** — Snowflake auto-determines the refresh schedule to meet your freshness SLA
> - **Open format output** — data is written as Iceberg, readable by Spark, DuckDB, Trino, etc.
> - **No external orchestration** — no Airflow, no cron, no manual refresh triggers
> - **Snowflake Managed Storage** — Iceberg metadata and data files managed by Snowflake

Aggregates each player's **total score**, **bonus pops** (favorite-color matches), and **last event timestamp**. This is the go-to table for ranking players.

**Try it with Cortex Code:** Click the SQL cell below, press **Cmd+K** / **Ctrl+K**, and paste this prompt:

> Create a Dynamic Iceberg Table called dt_player_leaderboard in the silver schema that aggregates total score, bonus pop count, and last event timestamp per player from the bronze balloon game events table. Use Snowflake Managed storage (no BASE_LOCATION). Reference the Variables cell for warehouse, external volume, target lag, database name, and bronze table reference — expand all variables to their actual values in the SQL. Parse the JSON event column to extract player, score, favorite_color_bonus, and event_ts fields.

In [ ]:
%%sql -r dt_leaderboard_result
-- TODO update with the generated SQL for create dt_player_leaderboard

---

## Step 4: Balloon Color Stats (`dt_balloon_color_stats`)

Breaks down each player's performance **by balloon color** — pops, points, and bonus hits. Useful for analyzing color preferences and strategies.

In [ ]:
%%sql -r dt_color_stats_result
CREATE OR REPLACE DYNAMIC ICEBERG TABLE {{DB_NAME}}.silver.dt_balloon_color_stats (
  player STRING,
  balloon_color STRING,
  balloon_pops NUMBER(38,0),
  points_by_color NUMBER(38,0),
  bonus_hits NUMBER(38,0),
  last_event_ts TIMESTAMP_NTZ
)
  TARGET_LAG = '{{TARGET_LAG}}'
  WAREHOUSE = {{WAREHOUSE}}
  EXTERNAL_VOLUME = '{{EXTERNAL_VOLUME}}'
  CATALOG = 'SNOWFLAKE'
AS
SELECT
  e.player,
  e.balloon_color,
  COUNT(*) AS balloon_pops,
  SUM(e.score_i) AS points_by_color,
  COUNT_IF(e.fav_bonus) AS bonus_hits,
  MAX(e.ts) AS last_event_ts
FROM (
  SELECT
    v:player::STRING AS player,
    v:balloon_color::STRING AS balloon_color,
    v:score::INTEGER AS score_i,
    v:favorite_color_bonus::BOOLEAN AS fav_bonus,
    v:event_ts::TIMESTAMP_NTZ AS ts
  FROM (
    SELECT PARSE_JSON(event) AS v
    FROM {{BRONZE_ICEBERG_TABLE}}
  ) q
) e
GROUP BY e.player, e.balloon_color;

---

## Step 5: Real-Time Scores (`dt_realtime_scores`)

Computes player scores in **15-second sliding windows** using `TIME_SLICE`. Shows how scores accumulate in near-real-time micro-batches.

In [ ]:
%%sql -r dt_realtime_result
CREATE OR REPLACE DYNAMIC ICEBERG TABLE {{DB_NAME}}.silver.dt_realtime_scores (
  player STRING,
  total_score NUMBER(38,0),
  window_start TIMESTAMP_NTZ,
  window_end TIMESTAMP_NTZ
)
  TARGET_LAG = '{{TARGET_LAG}}'
  WAREHOUSE = {{WAREHOUSE}}
  EXTERNAL_VOLUME = '{{EXTERNAL_VOLUME}}'
  CATALOG = 'SNOWFLAKE'
AS
SELECT
  w.player,
  w.total_score,
  w.window_start,
  DATEADD(second, 15, w.window_start) AS window_end
FROM (
  SELECT
    e.player,
    SUM(e.score_i) AS total_score,
    TIME_SLICE(e.ts, 15, 'SECOND') AS window_start
  FROM (
    SELECT
      v:player::STRING AS player,
      v:balloon_color::STRING AS balloon_color,
      v:score::INTEGER AS score_i,
      v:favorite_color_bonus::BOOLEAN AS fav_bonus,
      v:event_ts::TIMESTAMP_NTZ AS ts
    FROM (
      SELECT PARSE_JSON(event) AS v
      FROM {{BRONZE_ICEBERG_TABLE}}
    ) q
  ) e
  GROUP BY e.player, TIME_SLICE(e.ts, 15, 'SECOND')
) w;

---

## Step 6: Balloon Colored Pops (`dt_balloon_colored_pops`)

Combines per-player, per-color breakdown **with 15-second time windows**. Gives the most granular view of who popped what color and when.

In [ ]:
%%sql -r dt_colored_pops_result
CREATE OR REPLACE DYNAMIC ICEBERG TABLE {{DB_NAME}}.silver.dt_balloon_colored_pops (
  player STRING,
  balloon_color STRING,
  balloon_pops NUMBER(38,0),
  points_by_color NUMBER(38,0),
  bonus_hits NUMBER(38,0),
  window_start TIMESTAMP_NTZ,
  window_end TIMESTAMP_NTZ
)
  TARGET_LAG = '{{TARGET_LAG}}'
  WAREHOUSE = {{WAREHOUSE}}
  EXTERNAL_VOLUME = '{{EXTERNAL_VOLUME}}'
  CATALOG = 'SNOWFLAKE'
AS
SELECT
  w.player,
  w.balloon_color,
  w.balloon_pops,
  w.points_by_color,
  w.bonus_hits,
  w.window_start,
  DATEADD(second, 15, w.window_start) AS window_end
FROM (
  SELECT
    e.player,
    e.balloon_color,
    COUNT(*) AS balloon_pops,
    SUM(e.score_i) AS points_by_color,
    COUNT_IF(e.fav_bonus) AS bonus_hits,
    TIME_SLICE(e.ts, 15, 'SECOND') AS window_start
  FROM (
    SELECT
      v:player::STRING AS player,
      v:balloon_color::STRING AS balloon_color,
      v:score::INTEGER AS score_i,
      v:favorite_color_bonus::BOOLEAN AS fav_bonus,
      v:event_ts::TIMESTAMP_NTZ AS ts
    FROM (
      SELECT PARSE_JSON(event) AS v
      FROM {{BRONZE_ICEBERG_TABLE}}
    ) q
  ) e
  GROUP BY e.player, e.balloon_color, TIME_SLICE(e.ts, 15, 'SECOND')
) w;

---

## Step 7: Color Performance Trends (`dt_color_performance_trends`)

Tracks **average score per pop** and **total pops by balloon color** across 15-second windows. Reveals which colors are most rewarding over time.

In [ ]:
%%sql -r dt_trends_result
CREATE OR REPLACE DYNAMIC ICEBERG TABLE {{DB_NAME}}.silver.dt_color_performance_trends (
  balloon_color STRING,
  avg_score_per_pop NUMBER(38,6),
  total_pops NUMBER(38,0),
  window_start TIMESTAMP_NTZ,
  window_end TIMESTAMP_NTZ
)
  TARGET_LAG = '{{TARGET_LAG}}'
  WAREHOUSE = {{WAREHOUSE}}
  EXTERNAL_VOLUME = '{{EXTERNAL_VOLUME}}'
  CATALOG = 'SNOWFLAKE'
AS
SELECT
  w.balloon_color,
  w.avg_score_per_pop,
  w.total_pops,
  w.window_start,
  DATEADD(second, 15, w.window_start) AS window_end
FROM (
  SELECT
    e.balloon_color,
    AVG(e.score_i) AS avg_score_per_pop,
    COUNT(*) AS total_pops,
    TIME_SLICE(e.ts, 15, 'SECOND') AS window_start
  FROM (
    SELECT
      v:player::STRING AS player,
      v:balloon_color::STRING AS balloon_color,
      v:score::INTEGER AS score_i,
      v:favorite_color_bonus::BOOLEAN AS fav_bonus,
      v:event_ts::TIMESTAMP_NTZ AS ts
    FROM (
      SELECT PARSE_JSON(event) AS v
      FROM {{BRONZE_ICEBERG_TABLE}}
    ) q
  ) e
  GROUP BY e.balloon_color, TIME_SLICE(e.ts, 15, 'SECOND')
) w;

---

## Step 8: Verify Dynamic Tables

Before querying, check that all five DTs have completed their initial refresh.

**Use Cortex Code:** Ask Cortex Code chat:

> Show me the scheduling state, last completed refresh time, and target lag for all dynamic tables matching 'dt_%' in the silver schema. Use the database name from the Variables cell.

Or run the pre-filled check below. Wait until `scheduling_state` shows **ACTIVE** for all five tables before proceeding to the verify queries.

### 8a — Discovery: all five DTs visible and refreshing

In [ ]:
%%sql -r dt_verify_result
SHOW DYNAMIC TABLES LIKE 'dt_%' IN {{DB_NAME}}.silver;

### 8b — Player Leaderboard: top 15 players by score

In [ ]:
%%sql -r verify_leaderboard
SELECT player, total_score, bonus_pops, last_event_ts
FROM {{DB_NAME}}.silver.dt_player_leaderboard
ORDER BY total_score DESC NULLS LAST
LIMIT 5;

In [ ]:
%%sql -r verify_leaderboard_count
SELECT COUNT(*) AS leaderboard_rows FROM {{DB_NAME}}.silver.dt_player_leaderboard;

### 8c — Balloon Color Stats: player × color breakdown

In [ ]:
%%sql -r verify_color_stats
SELECT player, balloon_color, balloon_pops, points_by_color, bonus_hits, last_event_ts
FROM {{DB_NAME}}.silver.dt_balloon_color_stats
ORDER BY player, points_by_color DESC NULLS LAST
LIMIT 20;

### 8d — Real-Time Scores: 15-second windowed scores

In [ ]:
%%sql -r verify_realtime
SELECT player, total_score, window_start, window_end
FROM {{DB_NAME}}.silver.dt_realtime_scores
ORDER BY window_start DESC, player
LIMIT 20;

In [ ]:
%%sql -r verify_realtime_spans
SELECT
  COUNT(*) AS windowed_rows,
  COUNT_IF(window_end = DATEADD(second, 15, window_start)) AS rows_with_15s_span
FROM {{DB_NAME}}.silver.dt_realtime_scores;

### 8e — Balloon Colored Pops: window × player × color

In [ ]:
%%sql -r verify_colored_pops
SELECT player, balloon_color, balloon_pops, window_start, window_end
FROM {{DB_NAME}}.silver.dt_balloon_colored_pops
ORDER BY window_start DESC, player, balloon_color
LIMIT 20;

### 8f — Color Performance Trends: avg score per pop by color over time

In [ ]:
%%sql -r verify_trends
SELECT balloon_color, avg_score_per_pop, total_pops, window_start, window_end
FROM {{DB_NAME}}.silver.dt_color_performance_trends
ORDER BY window_start DESC, balloon_color
LIMIT 20;

### 8g — Bronze vs Silver Row-Count Parity

Compare distinct player counts between bronze and silver. After a full DT refresh, `dt_players` should equal `bronze_distinct_players`.

In [ ]:
%%sql -r verify_parity
WITH bronze AS (
  SELECT PARSE_JSON(event):player::STRING AS player
  FROM {{BRONZE_ICEBERG_TABLE}}
)
SELECT
  (SELECT COUNT(*) FROM {{DB_NAME}}.silver.dt_player_leaderboard) AS dt_players,
  (SELECT COUNT(DISTINCT player) FROM bronze) AS bronze_distinct_players;

---

## Step 9: Explore with Cortex Code

Now that all five silver tables are live, use **Cortex Code** to ask questions across them. Click the SQL cell below, press **Cmd+K** / **Ctrl+K**, and try one of these prompts:

> Which player improved the most between their earliest and latest time windows?

> What's each player's best performing balloon color and how does it compare to their overall average?

> Are there any balloon colors that consistently score higher than others across all players?

Or ask your own question — Cortex Code will figure out which silver tables to query and how to combine them.

---

## What Just Happened?

You built a **fully automated silver aggregation pipeline** using Dynamic Iceberg Tables:

| Layer | What | Format | Managed By |
|---|---|---|---|
| **Bronze** | Raw balloon game events | Iceberg (AWS Glue) | AWS / Lake Formation |
| **Silver** | 5 aggregation tables | Iceberg (Snowflake Managed Storage) | Snowflake Dynamic Tables |

| Silver Table | Aggregation |
|---|---|
| `dt_player_leaderboard` | Total score + bonus pops per player |
| `dt_balloon_color_stats` | Per-player, per-color breakdown |
| `dt_realtime_scores` | 15-second windowed scores |
| `dt_balloon_colored_pops` | Windowed scores by player + color |
| `dt_color_performance_trends` | Avg score per pop by color over time |

Key takeaways:
- **No orchestration** — Snowflake auto-refreshes based on `TARGET_LAG`
- **Open format** — silver data is Iceberg, readable by Spark, DuckDB, Trino, etc.
- **Declarative** — you defined *what* to compute, Snowflake handles *when* and *how*
- **Bronze stays untouched** — the CLD reads Glue Iceberg in place; silver writes to Snowflake Managed Storage

**Next up:** Make your silver data AI-ready — open [**si_lab_guide.ipynb**](https://github.com/Snowflake-Labs/sfguide-lakehouse-iceberg-production-pipelines/tree/main/notebooks/si_lab_guide.ipynb) to create a Semantic View and configure Snowflake Intelligence for natural-language querying of these silver tables.

---

## Cleanup

Drop the silver database to remove all Dynamic Iceberg Tables. DT refreshes stop automatically when the tables are dropped.

> **Note:** This does **not** delete the external volume or the underlying S3 data. Remove those separately via the AWS console or `task dt:extvol-delete` if needed.

In [ ]:
%%sql -r cleanup_result
DROP DATABASE IF EXISTS {{DB_NAME}};